<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/CNN_%ED%95%84%ED%84%B0_%EC%8B%9C%EA%B0%81%ED%99%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 📘 CNN 필터 시각화 - Google Colab용 노트북

# 🔧 필요한 라이브러리 설치
!pip install matplotlib numpy opencv-python

# 📦 라이브러리 임포트
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 📥 이미지 로드 (파일 업로드 필요 시 아래 코드 사용)
from google.colab import files
uploaded = files.upload()

import io
file_bytes = list(uploaded.values())[0]
image = cv2.imdecode(np.frombuffer(file_bytes, np.uint8), cv2.IMREAD_COLOR)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 📏 이미지 정보 출력
print(f"이미지 크기: {image.shape}")

# 🔍 필터 정의 (엣지 검출 등)
filters = {
    'Original': np.array([[0]]),
    'Edge Detection (Sobel X)': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    'Edge Detection (Sobel Y)': np.array([[-1,-2,-1],[0,0,0],[1,2,1]]),
    'Sharpen': np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]]),
    'Blur': (1/9) * np.ones((3,3))
}

# 🧠 CNN 필터 적용 함수
def apply_filter(image, kernel):
    return cv2.filter2D(src=image, ddepth=-1, kernel=kernel)

# 🎨 결과 시각화 함수
def plot_filters(image, filters):
    plt.figure(figsize=(15, 10))
    for i, (name, kernel) in enumerate(filters.items()):
        if name == 'Original':
            filtered = image
        else:
            filtered = apply_filter(image, kernel)
        plt.subplot(2, 3, i+1)
        plt.imshow(filtered)
        plt.title(name)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# 📊 필터 시각화 실행
plot_filters(image, filters)



In [ ]:

# 🔧 필요한 라이브러리 설치
!pip install opencv-python matplotlib

속도제한 표지판

🔍 검출 방법

빨간색 검출 - HSV 색공간에서 빨간색 추출
원형 검출 - HoughCircles로 원형 모양 찾기
조합 판정 - 빨간 원형 = 속도제한 표지판

In [ ]:
# 🚗 속도제한 표지판 간단 인식 - Google Colab용


# 📦 라이브러리 임포트
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io

# 📥 이미지 업로드
print("📷 속도제한 표지판 이미지를 업로드하세요!")
uploaded = files.upload()

# 이미지 로드 및 변환
file_bytes = list(uploaded.values())[0]
image = cv2.imdecode(np.frombuffer(file_bytes, np.uint8), cv2.IMREAD_COLOR)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print(f"이미지 크기: {image.shape}")

# 🎯 속도제한 표지판 검출 함수
def detect_speed_limit_sign(image):
    """빨간 원형 표지판 검출"""

    # HSV 색공간 변환
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    # 빨간색 범위 정의
    lower_red1 = np.array([0, 120, 70])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([170, 120, 70])
    upper_red2 = np.array([180, 255, 255])

    # 빨간색 마스크 생성
    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    red_mask = cv2.bitwise_or(mask1, mask2)

    # 원형 검출
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1, 20,
                              param1=50, param2=30, minRadius=10, maxRadius=100)

    results = []
    if circles is not None:
        circles = np.round(circles[0, :]).astype("int")
        for (x, y, r) in circles:
            # 원 영역에서 빨간색 비율 확인
            circle_mask = np.zeros(gray.shape, dtype=np.uint8)
            cv2.circle(circle_mask, (x, y), r, 255, -1)

            red_in_circle = cv2.bitwise_and(red_mask, circle_mask)
            red_ratio = cv2.countNonZero(red_in_circle) / cv2.countNonZero(circle_mask)

            if red_ratio > 0.3:  # 30% 이상 빨간색이면 속도제한 표지판으로 판정
                results.append({
                    'center': (x, y),
                    'radius': r,
                    'red_ratio': red_ratio,
                    'type': 'speed_limit'
                })

    return results, red_mask

# 🔍 표지판 검출 실행
signs, red_mask = detect_speed_limit_sign(image)

# 🎨 결과 시각화
def visualize_results(image, signs, red_mask):
    plt.figure(figsize=(15, 5))

    # 원본 이미지
    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    # 빨간색 마스크
    plt.subplot(1, 3, 2)
    plt.imshow(red_mask, cmap='gray')
    plt.title('Red Color Detection')
    plt.axis('off')

    # 검출 결과
    plt.subplot(1, 3, 3)
    result_image = image.copy()

    for sign in signs:
        x, y = sign['center']
        r = sign['radius']

        # 원 그리기
        cv2.circle(result_image, (x, y), r, (0, 255, 0), 3)
        cv2.circle(result_image, (x, y), 2, (0, 255, 0), 3)

        # 텍스트 추가
        cv2.putText(result_image, f'Speed Limit ({sign["red_ratio"]:.1%})',
                   (x-50, y-r-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    plt.imshow(result_image)
    plt.title(f'Detection Result: {len(signs)} signs found')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# 📊 결과 출력
visualize_results(image, signs, red_mask)

print(f"\n🎯 검출 결과:")
print(f"총 {len(signs)}개의 속도제한 표지판이 발견되었습니다!")

for i, sign in enumerate(signs):
    print(f"표지판 {i+1}: 중심({sign['center'][0]}, {sign['center'][1]}), "
          f"반지름 {sign['radius']}, 빨간색 비율 {sign['red_ratio']:.1%}")

# 🔍 간단한 숫자 인식 (옵션)
def simple_number_detection(image, signs):
    """검출된 원형 영역에서 숫자 추정"""
    for i, sign in enumerate(signs):
        x, y, r = sign['center'][0], sign['center'][1], sign['radius']

        # 원형 영역 추출
        crop_size = int(r * 1.5)
        x1, y1 = max(0, x-crop_size), max(0, y-crop_size)
        x2, y2 = min(image.shape[1], x+crop_size), min(image.shape[0], y+crop_size)

        cropped = image[y1:y2, x1:x2]

        # 간단한 크기 기반 추정
        area = np.pi * r * r
        if area < 500:
            speed = "30"
        elif area < 1000:
            speed = "50"
        elif area < 2000:
            speed = "80"
        else:
            speed = "100"

        print(f"표지판 {i+1} 추정 속도: {speed} km/h (크기 기준)")

# 숫자 인식 실행
if signs:
    simple_number_detection(image, signs)
else:
    print("❌ 속도제한 표지판이 검출되지 않았습니다.")
    print("💡 팁: 빨간 원형 테두리가 있는 속도제한 표지판을 업로드해보세요!")